# Strategy Development

Prototype simple strategies that plug straight into the bot's base classes.

In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import load_config
from src.utils.logger import get_logger
from src.data.data_manager import DataManager
from src.strategies.base_strategy import BaseStrategy

config = load_config()
logger = get_logger("notebooks.strategy_dev")
data_manager = DataManager(config=config)


Loading cofiguration from /home/ciprimarian/Repos/ Local Repos/tradingbot/src/config/trading_config.yaml...
Loading cofiguration from /home/ciprimarian/Repos/ Local Repos/tradingbot/src/config/trading_config.yaml...
2025-11-18 23:53:34,654 | INFO | src.data.market_data | Market Data handler initialized | base_url: https://data.alpaca.markets


## Load features

In [2]:
df = data_manager.load_data('engineered_features')
if df.empty:
    raise RuntimeError("Run 02_data_engineering.ipynb first to generate engineered features data.")

features = df.copy()
features.tail()

2025-11-18 23:53:42,941 | INFO | src.data.data_manager | Loading dataset engineered_features from data/processed/engineered_features.parquet


,open,high,low,close,volume,sma_20,sma_50,rsi_14,return_1d,lag_close_1,day_of_week
t,,,,,,,,,,,
2024-05-17 04:00:00+00:00,943.69,947.4,918.06,924.79,35989353,879.3725,882.4177,59.353459,-0.019924,943.59,4
2024-05-20 04:00:00+00:00,937.50,952.0,934.40,947.80,31876446,887.0035,883.8681,65.995876,0.024881,924.79,0
2024-05-21 04:00:00+00:00,935.99,954.0,931.80,953.86,32894646,893.4850,885.7905,76.341057,0.006394,947.80,1
2024-05-22 04:00:00+00:00,954.59,960.2,932.49,949.50,54865849,901.1215,886.3979,71.649362,-0.004571,953.86,2
2024-05-23 04:00:00+00:00,1020.28,1063.2,1015.20,1037.99,83506528,911.7050,888.9801,77.827215,0.093196,949.50,3


## Define a quick RSI threshold strategy

In [3]:
class RsiThresholdStrategy(BaseStrategy):
    def generate_signals(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.copy()
        df["signal"] = 0
        df.loc[df["rsi_14"] < 30, "signal"] = 1
        df.loc[df["rsi_14"] > 70, "signal"] = -1
        df["position"] = df["signal"].diff().fillna(0)
        return df

strategy = RsiThresholdStrategy()
signals = strategy.generate_signals(features)
signals[["close", "rsi_14", "signal", "position"]].tail()


,close,rsi_14,signal,position
t,,,,
2024-05-17 04:00:00+00:00,924.79,59.353459,0,0.0
2024-05-20 04:00:00+00:00,947.80,65.995876,0,0.0
2024-05-21 04:00:00+00:00,953.86,76.341057,-1,-1.0
2024-05-22 04:00:00+00:00,949.50,71.649362,-1,0.0
2024-05-23 04:00:00+00:00,1037.99,77.827215,-1,0.0
